For building, training, and using a recommended model, use `Surprise`.

In [ ]:
!pip install scikit-surprise

In [ ]:
!pip install "numpy<2"

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV

from surprise import accuracy

Read `ctgan_gym.xls` file.

In [ ]:
df = pd.read_csv('ctgan_gym.xls')

print(df.tail())

       program_name  day  muscle_group  sets  min_reps  max_reps  difficulty  \
12552             8    0             3     3        12        10           1   
12553             3   10            13     3        10        10           1   
12554             4    0             8     3        14        15           1   
12555             7    2            10     3         8        10           2   
12556             3   12            14     3         8        12           0   

       age  gender  experience_numeric  days_per_week  goal  start_weight_kg  \
12552   22       1                   3              6     3        80.595362   
12553   38       0                   1              4     0        57.604077   
12554   44       0                   1              6     0        80.684114   
12555   48       0                   3              6     0        85.923107   
12556   26       0                   3              4     1        55.161877   

       end_weight_kg  goal_achieved  u

In [ ]:
print(df.isnull().sum())

program_name          0
day                   0
muscle_group          0
sets                  0
min_reps              0
max_reps              0
difficulty            0
age                   0
gender                0
experience_numeric    0
days_per_week         0
goal                  0
start_weight_kg       0
end_weight_kg         0
goal_achieved         0
user_rating           0
dtype: int64


Prepare the dataset for `SVD` model.

In [ ]:
df['user_id'] = df.groupby(['age', 'gender', 'experience_numeric', 'days_per_week', 'goal']).ngroup()

In [ ]:
df['combined_score'] = (
    df['user_rating'] * 0.6 +
    (df['goal_achieved'] == 2).astype(int) * 3 +
    (df['goal_achieved'] == 1).astype(int) * 1 +
    (df['end_weight_kg'] > df['start_weight_kg']).astype(int) * 0.5
)

min_score = df['combined_score'].min()
max_score = df['combined_score'].max()
df['rating'] = 1 + 4 * (df['combined_score'] - min_score) / (max_score - min_score)
df['rating'] = df['rating'].round(1)

Use `Reader` for setting the rating limits from 1 to 5.  
`Dataset.load_from_df` is used to convert a DataFrame into the format required by `Surprise`.


In [ ]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'program_name', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

`SVD` is popular for building recommendation models

In [ ]:
model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)

model.fit(trainset)

In [ ]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.7225
MAE:  0.5378


Search for the best hyperparameters for the model


In [ ]:
param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [10, 20, 30],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.05, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

print("The best features:", gs.best_params['rmse'])
print("best RMSE:", gs.best_score['rmse'])

best_model = gs.best_estimator['rmse']

The best features: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.002, 'reg_all': 0.02}
best RMSE: 0.701050208246185


Evaluate the best model using cross-validation


In [ ]:
cv_results = cross_validate(best_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print(f"avg RMSE: {cv_results['test_rmse'].mean():.4f}")
print(f"Standard deviation RMSE: {cv_results['test_rmse'].std():.4f}")
print(f"avg MAE: {cv_results['test_mae'].mean():.4f}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.6952  0.7042  0.7074  0.7142  0.6888  0.7020  0.0090  
MAE (testset)     0.5217  0.5319  0.5387  0.5404  0.5197  0.5305  0.0085  
Fit time          0.14    0.15    0.15    0.15    0.15    0.15    0.00    
Test time         0.01    0.01    0.01    0.01    0.01    0.01    0.00    
avg RMSE: 0.7020
Standard deviation RMSE: 0.0090
avg MAE: 0.5305


Save the model

In [ ]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)